# Real-time Face Detection with Automatic Device Switching using OpenVINO

This notebook demonstrates how to build a real-time face detection application using Intel's OpenVINO toolkit with the **AUTO device plugin** — which automatically selects and switches between available inference devices (CPU, GPU, NPU) at runtime.

### What you will learn:
- How to load and run the `face-detection-0200` model with OpenVINO
- How to use the OpenVINO **AUTO plugin** for automatic device selection
- How to compare inference performance across CPU, GPU, and AUTO mode
- How to run real-time face detection on a webcam feed

### Use case:
AI PCs have multiple inference engines (CPU, GPU, NPU). This notebook shows how OpenVINO's AUTO plugin can intelligently select and switch between devices — making AI applications more efficient and adaptive.

---
**Related GSoC 2026 Project:** [Continuous Face-Detection with Automatic Device Switching on AI PCs](https://github.com/openvinotoolkit/openvino/wiki/Project-ideas-for-2026)

## Prerequisites

Install required packages:

In [ ]:
%pip install -q openvino opencv-python numpy matplotlib openvino-dev

## Step 1: Import Libraries and Check Available Devices

In [ ]:
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from pathlib import Path
import openvino as ov

# Initialize OpenVINO Core
core = ov.Core()

# Check available devices
available_devices = core.available_devices
print("Available inference devices:")
for device in available_devices:
    device_name = core.get_property(device, "FULL_DEVICE_NAME")
    print(f"  {device}: {device_name}")

print(f"\nTotal devices found: {len(available_devices)}")

## Step 2: Download the Face Detection Model

We use Intel's pre-trained `face-detection-0200` model from Open Model Zoo. This is a lightweight MobileNetV2-based face detector optimized for edge deployment.

In [ ]:
from pathlib import Path
import subprocess

MODEL_NAME = "face-detection-0200"
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

# Download model using omz_downloader
download_command = [
    "omz_downloader",
    "--name", MODEL_NAME,
    "--output_dir", str(MODEL_DIR),
    "--precision", "FP16"
]

print(f"Downloading {MODEL_NAME}...")
result = subprocess.run(download_command, capture_output=True, text=True)
print(result.stdout)

# Find the model XML file
model_path = list(MODEL_DIR.glob(f"**/{MODEL_NAME}.xml"))[0]
print(f"Model downloaded to: {model_path}")

## Step 3: Load the Model

We load the model and inspect its input/output shapes.

In [ ]:
# Load the model
model = core.read_model(model_path)

# Inspect model inputs and outputs
print("Model Information:")
print(f"  Input name:  {model.input().get_any_name()}")
print(f"  Input shape: {model.input().shape}")
print(f"  Output name: {model.output().get_any_name()}")
print(f"  Output shape: {model.output().shape}")

# Model expects: [1, 3, H, W] — batch, channels, height, width
INPUT_HEIGHT = model.input().shape[2]
INPUT_WIDTH  = model.input().shape[3]
print(f"\nExpected input size: {INPUT_WIDTH}x{INPUT_HEIGHT}")

## Step 4: Define Helper Functions

In [ ]:
def preprocess(frame, input_h, input_w):
    """Resize and reformat frame for model input."""
    resized = cv2.resize(frame, (input_w, input_h))
    # Convert HWC to NCHW format
    blob = np.expand_dims(resized.transpose(2, 0, 1), axis=0)
    return blob.astype(np.float32)


def draw_detections(frame, detections, threshold=0.5):
    """Draw bounding boxes around detected faces."""
    h, w = frame.shape[:2]
    face_count = 0

    for detection in detections[0][0]:
        confidence = float(detection[2])
        if confidence < threshold:
            continue

        x1 = int(detection[3] * w)
        y1 = int(detection[4] * h)
        x2 = int(detection[5] * w)
        y2 = int(detection[6] * h)

        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Draw confidence label
        label = f"{confidence:.0%}"
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        face_count += 1

    return frame, face_count


def run_inference_benchmark(compiled_model, test_image, n_runs=100):
    """Run inference n times and return average latency in milliseconds."""
    infer_request = compiled_model.create_infer_request()
    blob = preprocess(test_image, INPUT_HEIGHT, INPUT_WIDTH)

    # Warm up
    for _ in range(5):
        infer_request.infer({0: blob})

    # Benchmark
    start = time.perf_counter()
    for _ in range(n_runs):
        infer_request.infer({0: blob})
    end = time.perf_counter()

    avg_latency_ms = (end - start) / n_runs * 1000
    return avg_latency_ms


print("Helper functions defined successfully!")

## Step 5: Performance Benchmark — CPU vs AUTO

Let's compare inference latency across different devices.

The **AUTO plugin** starts inference on CPU immediately (low compile time), then transparently moves to the best available accelerator (GPU/NPU) in the background — giving you the best of both worlds.

In [ ]:
# Create a test image (or load your own)
test_image = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

# Devices to benchmark
devices_to_test = ["CPU"]
if "GPU" in available_devices:
    devices_to_test.append("GPU")
devices_to_test.append("AUTO")

results = {}
N_RUNS = 100

for device in devices_to_test:
    print(f"Benchmarking on {device}...", end=" ", flush=True)
    try:
        compiled = core.compile_model(model, device)
        latency = run_inference_benchmark(compiled, test_image, N_RUNS)
        fps = 1000 / latency
        results[device] = {"latency_ms": latency, "fps": fps}
        print(f"Latency: {latency:.2f}ms | FPS: {fps:.1f}")
    except Exception as e:
        print(f"Skipped ({e})")

print("\nBenchmark complete!")

## Step 6: Visualize Benchmark Results

In [ ]:
if results:
    devices = list(results.keys())
    latencies = [results[d]["latency_ms"] for d in devices]
    fps_values = [results[d]["fps"] for d in devices]

    colors = ["#1565C0", "#00796B", "#E65100"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("OpenVINO Face Detection — Device Performance Comparison",
                 fontsize=14, fontweight="bold")

    # Latency chart
    bars1 = ax1.bar(devices, latencies, color=colors[:len(devices)], width=0.5)
    ax1.set_title("Inference Latency (lower is better)")
    ax1.set_ylabel("Latency (ms)")
    ax1.set_xlabel("Device")
    for bar, val in zip(bars1, latencies):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1f}ms", ha="center", fontweight="bold")

    # FPS chart
    bars2 = ax2.bar(devices, fps_values, color=colors[:len(devices)], width=0.5)
    ax2.set_title("Throughput (higher is better)")
    ax2.set_ylabel("FPS")
    ax2.set_xlabel("Device")
    for bar, val in zip(bars2, fps_values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1f}", ha="center", fontweight="bold")

    plt.tight_layout()
    plt.savefig("benchmark_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved as benchmark_results.png")

## Step 7: How the AUTO Plugin Works

The OpenVINO AUTO plugin works as follows:

1. **Startup**: Immediately begins inference on CPU (fastest to compile)
2. **Background loading**: Simultaneously compiles the model for GPU/NPU
3. **Transparent switch**: Once the better device is ready, inference moves there automatically
4. **No interruption**: The switch happens seamlessly — no frames are dropped

```python
# Simple usage — just change the device name!
compiled_model = core.compile_model(model, "AUTO")        # Automatic
compiled_model = core.compile_model(model, "CPU")         # Force CPU
compiled_model = core.compile_model(model, "AUTO:GPU,CPU") # Prefer GPU
compiled_model = core.compile_model(model, "AUTO:-CPU")   # Exclude CPU
```

## Step 8: Run Face Detection with AUTO Device

Now let's run face detection on a sample image using the AUTO plugin.

In [ ]:
# Compile model with AUTO device
compiled_model = core.compile_model(model, "AUTO")
infer_request = compiled_model.create_infer_request()

# Check which device is actually being used
exec_device = compiled_model.get_property("EXECUTION_DEVICES")
print(f"Inference running on: {exec_device}")

# Download a sample image
import urllib.request
sample_url = "https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/image/coco_bike.jpg"
urllib.request.urlretrieve(sample_url, "sample.jpg")
frame = cv2.imread("sample.jpg")

# Run inference
blob = preprocess(frame, INPUT_HEIGHT, INPUT_WIDTH)
start = time.perf_counter()
infer_request.infer({0: blob})
end = time.perf_counter()

detections = infer_request.get_output_tensor(0).data
result_frame, face_count = draw_detections(frame.copy(), detections)

latency_ms = (end - start) * 1000
print(f"Inference latency: {latency_ms:.2f}ms")
print(f"Faces detected: {face_count}")

# Display result
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(result_frame, cv2.COLOR_BGR2RGB))
plt.title(f"Face Detection Result | Device: {exec_device} | "
          f"Latency: {latency_ms:.1f}ms | Faces: {face_count}")
plt.axis("off")
plt.tight_layout()
plt.show()

## Step 9: Live Webcam Demo (Optional)

Run this cell to start real-time face detection using your webcam. Press **Q** to quit.

In [ ]:
# Uncomment to run live webcam demo
# NOTE: This requires a webcam and a local environment (not Google Colab)

# cap = cv2.VideoCapture(0)
# if not cap.isOpened():
#     print("Webcam not available")
# else:
#     prev_time = time.time()
#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break
#
#         blob = preprocess(frame, INPUT_HEIGHT, INPUT_WIDTH)
#         infer_request.infer({0: blob})
#         detections = infer_request.get_output_tensor(0).data
#         frame, face_count = draw_detections(frame, detections)
#
#         curr_time = time.time()
#         fps = 1.0 / (curr_time - prev_time)
#         prev_time = curr_time
#
#         cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
#         cv2.putText(frame, f"Device: {exec_device}", (10, 60),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)
#         cv2.putText(frame, f"Faces: {face_count}", (10, 90),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)
#
#         cv2.imshow("OpenVINO Face Detection - AUTO", frame)
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break
#
#     cap.release()
#     cv2.destroyAllWindows()

print("Webcam demo code is ready — uncomment to run!")

## Summary

In this notebook we covered:

| Topic | What we did |
|---|---|
| **Model** | Loaded Intel's `face-detection-0200` pretrained model |
| **AUTO plugin** | Used OpenVINO AUTO for automatic device selection |
| **Benchmark** | Compared latency and FPS across CPU, GPU, and AUTO |
| **Detection** | Ran face detection on a sample image |
| **Live demo** | Provided webcam inference code |

### Key Takeaway
The OpenVINO AUTO plugin makes it easy to write device-agnostic AI applications. Your code doesn't need to know whether the user has a GPU or NPU — AUTO handles it automatically, always choosing the best available device.

### Next Steps
- Explore [OpenVINO AUTO plugin documentation](https://docs.openvino.ai/2024/openvino-workflow/running-inference/inference-devices-and-modes/auto-device-selection.html)
- Try with other models from [Open Model Zoo](https://github.com/openvinotoolkit/open_model_zoo)
- Extend this notebook with runtime device switching based on system load

---
*Author: Saeed Fahim | GitHub: [ffaahhimm](https://github.com/ffaahhimm) | GSoC 2026 Contributor*